In [ ]:
# ============================================================
# Cell 0: Install dependencies for SmolVLM-500M
# ============================================================
!pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.13.0 datasets==3.0.1
print("Dependencies installed.")

In [ ]:
# ============================================================
# Cell 1: Imports and Reproducibility
# ============================================================
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
from PIL import Image

from google.colab import drive

# SmolVLM uses standard AutoProcessor + AutoModelForVision2Seq
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print(f"Torch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Cell 2: Mount Drive and Configure Paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

BASE_DRIVE_PATH = "/content/drive/MyDrive/Kaggle_ScienceQA"

TRAIN_CSV_PATH      = os.path.join(BASE_DRIVE_PATH, "train.csv")
VAL_CSV_PATH        = os.path.join(BASE_DRIVE_PATH, "val.csv")
# Unseen validation subset: val rows whose questions never appear in train.
# Used to monitor true generalization performance.
VAL_UNSEEN_CSV_PATH = os.path.join(BASE_DRIVE_PATH, "val_unseen.csv")

# Raw image directories (no offline padding; padding is done at __getitem__ time)
IMAGES_DIR     = os.path.join(BASE_DRIVE_PATH, "train")
VAL_IMAGES_DIR = os.path.join(BASE_DRIVE_PATH, "val")

OUTPUT_DIR = os.path.join(BASE_DRIVE_PATH, "smolvlm_lora_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Cell 3: Dataset for SmolVLM (Truncation + Explicit Prompt + Dynamic Pad)
# ============================================================
import ast
import json
import pandas as pd
import os
import torch
from PIL import Image
from transformers import AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

CHOICE_LETTERS = "ABCDEFGHIJ"
MAX_LECTURE_CHARS = 1200
MAX_HINT_CHARS = 600

def parse_choices(raw):
    if isinstance(raw, list): return raw
    s = str(raw)
    try: return json.loads(s)
    except:
        try: return ast.literal_eval(s)
        except: return []

# Pad image to square (white background) to prevent SmolVLM's internal
# crop from losing edge information.
def pad_to_square(img, fill_color=(255, 255, 255)):
    w, h = img.size
    if w == h: return img
    side = max(w, h)
    new_img = Image.new("RGB", (side, side), fill_color)
    new_img.paste(img, ((side - w) // 2, (side - h) // 2))
    return new_img

def build_user_prompt(row):
    context_parts = []
    if 'lecture' in row and pd.notna(row['lecture']) and str(row['lecture']).strip():
        lec = str(row['lecture']).strip()
        if len(lec) > MAX_LECTURE_CHARS:
            half = MAX_LECTURE_CHARS // 2
            lec = lec[:half] + " ... " + lec[-half:]
        context_parts.append(lec)

    if 'hint' in row and pd.notna(row['hint']) and str(row['hint']).strip():
        hint = str(row['hint']).strip()
        if len(hint) > MAX_HINT_CHARS:
            hint = hint[:MAX_HINT_CHARS] + " ..."
        context_parts.append(hint)

    context_str = "\n".join(context_parts)
    choices = parse_choices(row['choices'])
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = ""
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"

    last_letter = CHOICE_LETTERS[len(choices) - 1] if choices else 'A'
    prompt += f"Pick ONE letter from A to {last_letter}.\nAnswer:"

    return prompt, choices

class ScienceQADataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, image_dir, is_train=True):
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['question', 'choices', 'answer']).reset_index(drop=True)
        self.df = df
        self.image_dir = image_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['image_path']).split('/')[-1]
        img_path = os.path.join(self.image_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        image = pad_to_square(image)

        user_text, choices = build_user_prompt(row)
        answer_idx = int(row['answer'])
        answer_idx = max(0, min(answer_idx, len(choices) - 1)) if choices else 0
        target_letter = CHOICE_LETTERS[answer_idx]

        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]},
            {"role": "assistant", "content": [{"type": "text", "text": f" {target_letter}"}]},
        ]

        return {"messages": messages, "image": image, "target_letter": target_letter}

print("Loading train / val datasets...")
train_dataset       = ScienceQADataset(TRAIN_CSV_PATH, IMAGES_DIR, is_train=True)
val_overall_dataset = ScienceQADataset(VAL_CSV_PATH, VAL_IMAGES_DIR, is_train=False)
val_unseen_dataset  = ScienceQADataset(VAL_UNSEEN_CSV_PATH, VAL_IMAGES_DIR, is_train=False)

print(f"Train: {len(train_dataset)} | Val (overall): {len(val_overall_dataset)} | Val (unseen): {len(val_unseen_dataset)}")

In [ ]:
# ============================================================
# Cell 4: SmolVLM DataCollator with Label Masking
# ============================================================
# Strategy: build two parallel views of every sample:
#   (a) full   = chat_template(user + assistant)   -> input_ids for the model
#   (b) prompt = chat_template(user only)          -> length used to locate the
#                                                     prefix that should be masked
# Both go through the processor with images so image-token expansion is
# accounted for. Labels are -100 on the prompt prefix and on padding,
# leaving only assistant tokens (the answer letter) contributing to loss.
class SmolVLMDataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.pad_id = processor.tokenizer.pad_token_id

    def __call__(self, batch):
        full_texts, prompt_texts, images = [], [], []
        for item in batch:
            msgs_full = item["messages"]
            msgs_prompt = [m for m in msgs_full if m["role"] != "assistant"]

            full_texts.append(self.processor.apply_chat_template(msgs_full, add_generation_prompt=False))
            prompt_texts.append(self.processor.apply_chat_template(msgs_prompt, add_generation_prompt=True))
            images.append([item["image"]])

        # Full pass (with assistant answer)
        inputs = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        # Prompt-only pass (also with images so image-token expansion length matches)
        prompt_inputs = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = inputs["input_ids"].clone()

        # Mask padding tokens
        labels[labels == self.pad_id] = -100

        # Mask the prompt prefix per-sample so only assistant tokens count for loss
        for i in range(len(batch)):
            prompt_len = (prompt_inputs["input_ids"][i] != self.pad_id).sum().item()
            valid_indices = (inputs["input_ids"][i] != self.pad_id).nonzero(as_tuple=True)[0]
            if len(valid_indices) >= prompt_len:
                prompt_indices = valid_indices[:prompt_len]
                labels[i, prompt_indices] = -100

        inputs["labels"] = labels
        return inputs

collator = SmolVLMDataCollator(processor)

# Sanity check: confirm only the assistant answer tokens receive gradient
sample_batch = collator([train_dataset[0], train_dataset[1]])
print("input_ids shape:", sample_batch["input_ids"].shape)
print("labels   shape:", sample_batch["labels"].shape)
n_active = (sample_batch["labels"] != -100).sum().item()
n_total  = sample_batch["labels"].numel()
print(f"Active loss tokens: {n_active}/{n_total}  ({n_active/n_total:.2%})")
print("Decoded non-masked labels (should contain only the answer letter and EOS):")
for i in range(sample_batch["labels"].size(0)):
    ids = sample_batch["labels"][i]
    visible = ids[ids != -100]
    print(f"  Sample {i}: {processor.tokenizer.decode(visible)!r}")

In [ ]:
# ============================================================
# Cell 5: Load SmolVLM-500M and apply LoRA (trainable params < 5M)
# ============================================================
print("Loading SmolVLM-500M-Instruct...")
# A 500M model fits on a single T4 in fp16; 4-bit quantization is unnecessary.
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# Required for gradient flow through frozen vision/text backbone with LoRA
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# LoRA configuration: r=16 over Q/K/V/O attention projections.
# Spreading capacity across four matrices (vs only Q/V) improves
# generalization while staying within the 5M trainable-parameter limit.
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Hard guardrail: the competition limits trainable parameters to 5,000,000.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameter audit: {trainable:,}")
assert trainable < 5_000_000, (
    f"Limit exceeded: {trainable:,} > 5,000,000. Reduce r or shrink target_modules."
)
print(f"Within 5M limit.")

# List which modules LoRA actually attached to (vision vs LLM split)
print("\nLoRA modules attached:")
lora_modules = [n for n, p in model.named_parameters()
                if p.requires_grad and "lora" in n.lower()]
vision_count = sum(1 for n in lora_modules if "vision" in n.lower() or "siglip" in n.lower())
llm_count = len(lora_modules) - vision_count
print(f"   LLM    LoRA modules: {llm_count}")
print(f"   Vision LoRA modules: {vision_count}")
print(f"   First 5 examples:    {lora_modules[:5]}")

if vision_count == 0:
    print("Note: vision encoder not covered by LoRA. Module names there may differ "
          "(e.g. 'query'/'value'). Add such aliases to target_modules if desired.")


In [ ]:
# ============================================================
# Cell 6: TrainingArguments + BalancedTrainer + Custom Early Stopping
# ============================================================
from torch.utils.data import WeightedRandomSampler
from transformers import TrainerCallback

# Force fp16 to guarantee identical numerics between Colab (training) and
# Kaggle T4 (offline inference). bf16 is disabled regardless of GPU.
use_bf16 = False
use_fp16 = True
print(f"Precision: bf16={use_bf16}, fp16={use_fp16}")


class BalancedTrainer(Trainer):
    """Sampler that gives equal weight to each num_choices bucket.

    Without this, the 5-choice questions (only ~3.5% of train) get drowned out
    by 2/3-choice questions and the model never learns to pick option E.
    """
    def _get_train_sampler(self, dataset=None):
        current_dataset = dataset if dataset is not None else self.train_dataset
        df = current_dataset.df
        nc_counts = df['choices'].apply(lambda x: len(parse_choices(x))).value_counts().to_dict()
        weights = [1.0 / nc_counts[len(parse_choices(row['choices']))]
                   for _, row in df.iterrows()]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        return sampler


class UnseenEarlyStoppingCallback(TrainerCallback):
    """Custom early stopping that tracks eval_unseen_loss reliably.

    The built-in EarlyStoppingCallback misbehaves when eval_dataset is a dict
    because it is invoked once per sub-dataset, and the metrics dict it sees
    contains only the keys for the current sub-eval. This callback instead
    inspects state.log_history at the end of every epoch (after all sub-evals
    have logged) and decides whether to stop.
    """
    def __init__(self, patience: int = 1, threshold: float = 0.0):
        self.patience = patience
        self.threshold = threshold
        self.bad_epochs = 0
        self.best = None

    def _get_unseen_history(self, state):
        return [entry["eval_unseen_loss"]
                for entry in state.log_history
                if "eval_unseen_loss" in entry]

    def on_epoch_end(self, args, state, control, **kwargs):
        history = self._get_unseen_history(state)
        if not history:
            return control

        latest = history[-1]
        if self.best is None or latest < self.best - self.threshold:
            self.best = latest
            self.bad_epochs = 0
            print(f"[EarlyStopping] new best eval_unseen_loss = {latest:.4f}")
        else:
            self.bad_epochs += 1
            print(f"[EarlyStopping] eval_unseen_loss did not improve "
                  f"({latest:.4f} vs best {self.best:.4f}); "
                  f"patience {self.bad_epochs}/{self.patience}")
            if self.bad_epochs >= self.patience:
                print("[EarlyStopping] patience exhausted, stopping training.")
                control.should_training_stop = True
        return control


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    num_train_epochs=3,

    logging_steps=10,
    logging_first_step=True,
    report_to="none",
    disable_tqdm=False,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_unseen_loss",
    greater_is_better=False,
    save_total_limit=2,

    bf16=use_bf16,
    fp16=use_fp16,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    label_names=["labels"],
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset={
        "overall": val_overall_dataset,
        "unseen":  val_unseen_dataset,
    },
    data_collator=collator,
    callbacks=[UnseenEarlyStoppingCallback(patience=1)],
)

gc.collect()
torch.cuda.empty_cache()

print("Starting balanced sampling training (fp16)...")
trainer.train()

In [ ]:
# ============================================================
# Cell 7: Save best LoRA weights
# ============================================================
FINAL_SAVE_PATH = os.path.join(OUTPUT_DIR, "final_lora_weights")
os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

print(f"Saving to {FINAL_SAVE_PATH} ...")
trainer.model.save_pretrained(FINAL_SAVE_PATH)
processor.save_pretrained(FINAL_SAVE_PATH)
print("Training complete. LoRA weights persisted to disk.")

In [ ]:
import time
from google.colab import drive
from google.colab import runtime

print("Training and local save complete; preparing to sync.")

# Force-flush all in-memory buffers to Google Drive before terminating the
# runtime. Skipping this can leave the most recent checkpoint files orphaned.
print("Syncing weights to Google Drive (do not close the tab)...")
drive.flush_and_unmount()
print("Google Drive sync complete; files are durable.")

# Brief countdown before instance teardown
print("Releasing GPU instance to stop billing...")
for i in range(10, 0, -1):
    print(f"Disconnecting in {i}s...")
    time.sleep(1)

# Equivalent to clicking "Disconnect and delete runtime"
print("Done. Compute released.")
runtime.unassign()

In [ ]:
# ============================================================
# Cell 9: Quick inference sanity check
# ============================================================
# Verifies that:
#   1. The saved LoRA weights reload cleanly on top of the base model
#   2. The end-to-end inference pipeline runs without error
#   3. Output format matches what downstream Kaggle inference expects
# This must mirror the offline Kaggle inference notebook exactly; any
# mismatch surfaces here rather than at submission time.
from peft import PeftModel
from IPython.display import display

# transformers 5.0+ renamed AutoModelForVision2Seq -> AutoModelForImageTextToText.
# Fall back to the old name for transformers 4.x environments.
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText

print("Reloading base + LoRA...")
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
infer_model = PeftModel.from_pretrained(base, FINAL_SAVE_PATH)
infer_model.eval()
infer_processor = AutoProcessor.from_pretrained(FINAL_SAVE_PATH)
print("LoRA weights loaded successfully.")

# Sample 5 val examples (single-sample sanity checks have too much variance)
val_df = pd.read_csv(VAL_CSV_PATH).dropna(subset=['question','choices','answer']).reset_index(drop=True)
samples = val_df.sample(5, random_state=42).reset_index(drop=True)

import re
def parse_letter_to_idx(text, n_choices):
    """Mirror of the inference-time letter parser."""
    if not text:
        return -1
    valid = set(CHOICE_LETTERS[:n_choices])
    for ch in text.upper():
        if ch in valid:
            return CHOICE_LETTERS.index(ch)
    m = re.search(r'\d+', text)
    if m:
        v = int(m.group())
        return v if 0 <= v < n_choices else -1
    return -1

correct = 0
print("\n" + "=" * 60)
for i, sample_row in samples.iterrows():
    img_name = str(sample_row['image_path']).split('/')[-1]
    img_path = os.path.join(VAL_IMAGES_DIR, img_name)
    image = Image.open(img_path).convert("RGB")

    # Same pad_to_square as training/inference
    image = pad_to_square(image)

    user_text, choices = build_user_prompt(sample_row)
    n_choices = len(choices)

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_text},
        ]},
    ]
    prompt = infer_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = infer_processor(text=[prompt], images=[[image]], return_tensors="pt").to(infer_model.device)

    with torch.no_grad():
        out_ids = infer_model.generate(**inputs, max_new_tokens=8, do_sample=False)

    resp = infer_processor.batch_decode(
        out_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]

    true_idx = int(sample_row['answer'])
    pred_idx = parse_letter_to_idx(resp, n_choices)
    is_correct = (pred_idx == true_idx)
    correct += int(is_correct)

    print(f"\n--- Sample {i+1}/5 (id={sample_row['id']}) ---")
    display(image.resize((250, int(250 * image.size[1]/image.size[0]))))
    print(f"Question: {sample_row['question'][:100]}...")
    print(f"  Choices ({n_choices}): {[c[:30] for c in choices]}")
    print(f"  Truth:  {CHOICE_LETTERS[true_idx]} (idx={true_idx})")
    print(f"  Output: {resp!r}  -> idx={pred_idx} ({CHOICE_LETTERS[pred_idx] if 0<=pred_idx<n_choices else '?'})")
    print(f"  {'CORRECT' if is_correct else 'WRONG'}")

print("\n" + "=" * 60)
print(f"Sanity check accuracy: {correct}/5 = {correct*20}%")
print("=" * 60)

if correct == 0:
    print("All 5 wrong: the LoRA may not have loaded, or the prompt is misaligned with training. "
          "Inspect build_user_prompt.")
elif correct >= 3:
    print("Sanity check passed; LoRA artifact is ready for Kaggle inference.")
else:
    print("Partial pass: within normal sampling variance, proceed.")